In [68]:
import pandas as pd

df = pd.read_pickle(r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.pkl")
df = df.sort_values("time")
df.head(), df.tail(), df.shape

(                       time  volume    mid_o    mid_h    mid_l    mid_c  \
 0 2016-01-07 00:00:00+00:00     542  1.07764  1.07832  1.07744  1.07778   
 1 2016-01-07 01:00:00+00:00    3167  1.07776  1.08100  1.07748  1.08029   
 2 2016-01-07 02:00:00+00:00    1567  1.08026  1.08176  1.07996  1.08152   
 3 2016-01-07 03:00:00+00:00     914  1.08156  1.08257  1.08150  1.08187   
 4 2016-01-07 04:00:00+00:00     649  1.08190  1.08256  1.08156  1.08236   
 
      bid_o    bid_h    bid_l    bid_c    ask_o    ask_h    ask_l    ask_c  
 0  1.07757  1.07823  1.07735  1.07770  1.07772  1.07840  1.07752  1.07787  
 1  1.07768  1.08092  1.07740  1.08020  1.07784  1.08109  1.07756  1.08038  
 2  1.08018  1.08169  1.07987  1.08144  1.08035  1.08184  1.08005  1.08159  
 3  1.08147  1.08249  1.08142  1.08178  1.08164  1.08265  1.08157  1.08196  
 4  1.08182  1.08247  1.08147  1.08228  1.08199  1.08264  1.08163  1.08245  ,
                            time  volume    mid_o    mid_h    mid_l    mid_c  \

In [69]:
import sys

sys.path.append(r"C:\Users\admin\Desktop\Forex_algo\code")

In [70]:
from technicals.patterns import apply_patterns  # adjust if your module path is different

df_feat = df.copy()

# Returns
df_feat["ret_1"]  = df_feat["mid_c"].pct_change()
df_feat["ret_3"]  = df_feat["mid_c"].pct_change(3)
df_feat["ret_6"]  = df_feat["mid_c"].pct_change(6)

# Rolling stats
df_feat["ma_10"]  = df_feat["mid_c"].rolling(window=10).mean()
df_feat["std_10"] = df_feat["mid_c"].rolling(window=10).std()
df_feat["ma_20"]  = df_feat["mid_c"].rolling(window=20).mean()
df_feat["std_20"] = df_feat["mid_c"].rolling(window=20).std()

df_feat["dist_ma10"] = df_feat["mid_c"] - df_feat["ma_10"]
df_feat["dist_ma20"] = df_feat["mid_c"] - df_feat["ma_20"]

# Volume features
df_feat["vol_chg_1"] = df_feat["volume"].pct_change()
df_feat["vol_chg_3"] = df_feat["volume"].pct_change(3)

# Add candlestick patterns
df_feat = apply_patterns(df_feat)

df_feat.head()


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,...,body_perc_prev_2,HANGING_MAN,SHOOTING_STAR,SPINNING_TOP,MARUBOZU,ENGULFING,TWEEZER_TOP,TWEEZER_BOTTOM,MORNING_STAR,EVENING_STAR
0,2016-01-07 00:00:00+00:00,542,1.07764,1.07832,1.07744,1.07778,1.07757,1.07823,1.07735,1.07770,...,NaN,False,False,False,False,False,False,False,False,False
1,2016-01-07 01:00:00+00:00,3167,1.07776,1.08100,1.07748,1.08029,1.07768,1.08092,1.07740,1.08020,...,NaN,False,False,False,False,False,False,False,False,False
2,2016-01-07 02:00:00+00:00,1567,1.08026,1.08176,1.07996,1.08152,1.08018,1.08169,1.07987,1.08144,...,15.909091,False,False,False,False,False,False,False,False,False
3,2016-01-07 03:00:00+00:00,914,1.08156,1.08257,1.08150,1.08187,1.08147,1.08249,1.08142,1.08178,...,71.875000,False,False,False,False,False,False,False,False,False
4,2016-01-07 04:00:00+00:00,649,1.08190,1.08256,1.08156,1.08236,1.08182,1.08247,1.08147,1.08228,...,70.000000,False,False,False,False,False,False,False,False,False


In [71]:
df_feat["target"] = (df_feat["mid_c"].shift(-1) > df_feat["mid_c"]).astype(int)
df_feat = df_feat.dropna()
df_feat[["time", "mid_c", "target"]].head()


,time,mid_c,target
19,2016-01-07 19:00:00+00:00,1.09310,1
20,2016-01-07 20:00:00+00:00,1.09346,0
21,2016-01-07 21:00:00+00:00,1.09308,0
22,2016-01-07 22:00:00+00:00,1.09218,1
23,2016-01-07 23:00:00+00:00,1.09260,0


In [72]:
pattern_cols = [
    "HANGING_MAN", "SHOOTING_STAR", "SPINNING_TOP", "MARUBOZU",
    "ENGULFING", "TWEEZER_TOP", "TWEEZER_BOTTOM",
    "MORNING_STAR", "EVENING_STAR"
]

feature_cols = [
    "ret_1", "ret_3", "ret_6",
    "ma_10", "std_10", "ma_20", "std_20",
    "dist_ma10", "dist_ma20",
    "vol_chg_1", "vol_chg_3",
] + pattern_cols

X = df_feat[feature_cols].copy().fillna(0)
y = df_feat["target"].copy()

X.shape, y.shape


((61436, 20), (61436,))

In [73]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print(classification_report(y_val, y_pred, digits=3))


              precision    recall  f1-score   support

           0      0.515     0.547     0.530      6154
           1      0.515     0.483     0.498      6134

    accuracy                          0.515     12288
   macro avg      0.515     0.515     0.514     12288
weighted avg      0.515     0.515     0.514     12288



In [74]:
horizon = 3  # predict 3 candles ahead
threshold = 0.001  # 0.1% move (adjust based on spread / TP)

future_ret = (df_feat["mid_c"].shift(-horizon) - df_feat["mid_c"]) / df_feat["mid_c"]

# 1 = strong up move, 0 = strong down/flat
df_feat["target"] = (future_ret > threshold).astype(int)

df_feat = df_feat.dropna()


In [75]:
X = df_feat[feature_cols].copy().fillna(0)
y = df_feat["target"].copy()


In [76]:
y.value_counts(normalize=True)


target
0    0.809753
1    0.190247
Name: proportion, dtype: float64

In [77]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)


In [78]:
threshold = 0.001  # 0.1%


In [35]:
threshold = 0.0005   # 0.05%


In [79]:
future_ret = (df_feat["mid_c"].shift(-horizon) - df_feat["mid_c"]) / df_feat["mid_c"]
df_feat["target"] = (future_ret > threshold).astype(int)


In [80]:
df_feat["target"].value_counts(normalize=True)


target
0    0.809753
1    0.190247
Name: proportion, dtype: float64

In [81]:
horizon = 3  # 3 hours


In [82]:
df_feat["ret_12"] = df_feat["mid_c"].pct_change(12)
df_feat["ret_24"] = df_feat["mid_c"].pct_change(24)

df_feat["ma_50"] = df_feat["mid_c"].rolling(50).mean()
df_feat["dist_ma50"] = df_feat["mid_c"] - df_feat["ma_50"]

df_feat["vol_10"] = df_feat["ret_1"].rolling(10).std()
df_feat["vol_20"] = df_feat["ret_1"].rolling(20).std()


In [83]:
pattern_cols = [
    "HANGING_MAN", "SHOOTING_STAR", "SPINNING_TOP", "MARUBOZU",
    "ENGULFING", "TWEEZER_TOP", "TWEEZER_BOTTOM",
    "MORNING_STAR", "EVENING_STAR"
]

feature_cols = [
    "ret_1", "ret_3", "ret_6",
    "ret_12", "ret_24",
    "ma_10", "std_10",
    "ma_20", "std_20",
    "ma_50",
    "dist_ma10", "dist_ma20", "dist_ma50",
    "vol_chg_1", "vol_chg_3",
    "vol_10", "vol_20",
] + pattern_cols


In [84]:
df_feat["ret_12"] = df_feat["mid_c"].pct_change(12)
df_feat["ret_24"] = df_feat["mid_c"].pct_change(24)

df_feat["ma_50"] = df_feat["mid_c"].rolling(50).mean()
df_feat["dist_ma50"] = df_feat["mid_c"] - df_feat["ma_50"]

df_feat["vol_10"] = df_feat["ret_1"].rolling(10).std()
df_feat["vol_20"] = df_feat["ret_1"].rolling(20).std()


In [85]:
X = df_feat[feature_cols].fillna(0)
y = df_feat["target"]


In [86]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print(classification_report(y_val, y_pred, digits=3))


              precision    recall  f1-score   support

           0      0.846     0.601     0.703     10055
           1      0.221     0.508     0.308      2233

    accuracy                          0.584     12288
   macro avg      0.533     0.555     0.505     12288
weighted avg      0.733     0.584     0.631     12288



In [44]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# You already did this:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
model.fit(X_train, y_train)

# Now get probabilities on validation
probs_val = model.predict_proba(X_val)[:, 1]  # probability of class 1 (big up move)


In [87]:
horizon = 6            # or your chosen value
threshold = 0.0005     # or your chosen value

future_ret = (df_feat["mid_c"].shift(-horizon) - df_feat["mid_c"]) / df_feat["mid_c"]
df_feat["target"] = (future_ret > threshold).astype(int)
df_feat = df_feat.dropna()


In [88]:
df_feat = df.copy()

# Returns
df_feat["ret_1"]  = df_feat["mid_c"].pct_change()
df_feat["ret_3"]  = df_feat["mid_c"].pct_change(3)
df_feat["ret_6"]  = df_feat["mid_c"].pct_change(6)

df_feat["ret_12"] = df_feat["mid_c"].pct_change(12)
df_feat["ret_24"] = df_feat["mid_c"].pct_change(24)

# MAs
df_feat["ma_10"]  = df_feat["mid_c"].rolling(10).mean()
df_feat["ma_20"]  = df_feat["mid_c"].rolling(20).mean()
df_feat["ma_50"]  = df_feat["mid_c"].rolling(50).mean()

# STD (these were missing!)
df_feat["std_10"] = df_feat["mid_c"].rolling(10).std()
df_feat["std_20"] = df_feat["mid_c"].rolling(20).std()

df_feat["dist_ma10"] = df_feat["mid_c"] - df_feat["ma_10"]
df_feat["dist_ma20"] = df_feat["mid_c"] - df_feat["ma_20"]
df_feat["dist_ma50"] = df_feat["mid_c"] - df_feat["ma_50"]

# Volatility / volume
df_feat["vol_chg_1"] = df_feat["volume"].pct_change()
df_feat["vol_chg_3"] = df_feat["volume"].pct_change(3)

df_feat["vol_10"] = df_feat["ret_1"].rolling(10).std()
df_feat["vol_20"] = df_feat["ret_1"].rolling(20).std()

# Patterns
df_feat = apply_patterns(df_feat)

# Future return + target
horizon = 6
threshold = 0.0005

future_ret = (df_feat["mid_c"].shift(-horizon) - df_feat["mid_c"]) / df_feat["mid_c"]
df_feat["future_ret"] = future_ret
df_feat["target"] = (df_feat["future_ret"] > threshold).astype(int)

# Clean
df_feat = df_feat.dropna().copy()
df_feat.reset_index(drop=True, inplace=True)


In [89]:
pattern_cols = [
    "HANGING_MAN", "SHOOTING_STAR", "SPINNING_TOP", "MARUBOZU",
    "ENGULFING", "TWEEZER_TOP", "TWEEZER_BOTTOM",
    "MORNING_STAR", "EVENING_STAR"
]

feature_cols = [
    "ret_1", "ret_3", "ret_6",
    "ret_12", "ret_24",
    "ma_10", "std_10",
    "ma_20", "std_20",
    "ma_50",
    "dist_ma10", "dist_ma20", "dist_ma50",
    "vol_chg_1", "vol_chg_3",
    "vol_10", "vol_20",
] + pattern_cols

X = df_feat[feature_cols].fillna(0)
y = df_feat["target"]


In [90]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, shuffle=False
)


In [91]:
probs_val = model.predict_proba(X_val)[:, 1]


In [92]:
df_ml_val = df_feat.iloc[len(X_train):].copy()


In [93]:
df_ml_val["p_up"] = probs_val


In [94]:
for thr in [0.6, 0.7, 0.8, 0.9]:
    subset = df_ml_val[df_ml_val["p_up"] >= thr]
    print(thr, len(subset), subset["future_ret"].mean(), subset["target"].mean())


0.6 617 -1.800266500616087e-05 0.41491085899513774
0.7 6 0.002091573619022393 0.8333333333333334
0.8 0 nan nan
0.9 0 nan nan


In [53]:
pip install xgboost


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [54]:
import xgboost as xgb
print(xgb.__version__)


1.7.6


In [55]:
conda activate base



Note: you may need to restart the kernel to use updated packages.



CondaError: Run 'conda init' before 'conda activate'



In [56]:
import sys
print(sys.executable)



c:\ProgramData\anaconda3\python.exe


In [57]:
pip install xgboost==1.7.6


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [58]:
import xgboost as xgb
print(xgb.__version__)


1.7.6


In [95]:
import xgboost as xgb

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "gpu_hist",     # GPU acceleration
    "predictor": "gpu_predictor",
    "max_depth": 8,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 1.0,
    "alpha": 0.5,
    "scale_pos_weight": (len(y) - y.sum()) / y.sum(),  # Handle imbalance
}


In [96]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

model = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=50,
    verbose_eval=50
)


[0]	train-logloss:0.69092	val-logloss:0.69276
[50]	train-logloss:0.62806	val-logloss:0.69270
[55]	train-logloss:0.62304	val-logloss:0.69294


In [97]:
probs_val = model.predict(dval)
y_pred = (probs_val >= 0.5).astype(int)


In [98]:
from sklearn.metrics import classification_report
print(classification_report(y_val, y_pred, digits=4))


              precision    recall  f1-score   support

           0     0.6468    0.5487    0.5937      7698
           1     0.3957    0.4965    0.4404      4582

    accuracy                         0.5292     12280
   macro avg     0.5212    0.5226    0.5171     12280
weighted avg     0.5531    0.5292    0.5365     12280



In [99]:
df_ml_val = df_feat.iloc[len(X_train):].copy()
df_ml_val["p_up"] = probs_val

for thr in [0.6, 0.7, 0.8, 0.9]:
    subset = df_ml_val[df_ml_val["p_up"] >= thr]
    print("Threshold:", thr)
    print("Samples:", len(subset))
    print("Mean future_ret:", subset["future_ret"].mean())
    print("Hit rate:", subset["target"].mean())
    print("------")


Threshold: 0.6
Samples: 823
Mean future_ret: 4.6651179328809985e-05
Hit rate: 0.425273390036452
------
Threshold: 0.7
Samples: 46
Mean future_ret: 0.0009060142644433284
Hit rate: 0.6521739130434783
------
Threshold: 0.8
Samples: 0
Mean future_ret: nan
Hit rate: nan
------
Threshold: 0.9
Samples: 0
Mean future_ret: nan
Hit rate: nan
------


In [100]:
import lightgbm as lgb

train_ds = lgb.Dataset(X_train, y_train)
val_ds   = lgb.Dataset(X_val, y_val)

params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "metric": "binary_logloss",
    "device": "gpu",          # GPU acceleration ON
    "gpu_platform_id": 0,
    "gpu_device_id": 0,
    
    "learning_rate": 0.03,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "min_data_in_leaf": 50,
}

model = lgb.train(
    params,
    train_ds,
    num_boost_round=2000,
    valid_sets=[train_ds, val_ds],
    callbacks=[
        lgb.log_evaluation(period=100),   # replaces verbose_eval
        lgb.early_stopping(stopping_rounds=100)  # replaces early_stopping_rounds
    ]
)


[LightGBM] [Info] Number of positive: 18368, number of negative: 30752
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 4345
[LightGBM] [Info] Number of data points in the train set: 49120, number of used features: 22
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4070 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 17 dense feature groups (0.94 MB) transferred to GPU in 0.002998 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.373941 -> initscore=-0.515345
[LightGBM] [Info] Start training from score -0.515345
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 17 dense feature groups (0.75 MB) transferred to GPU in 0.002518 secs. 1 sparse feature groups
Training until validation scores d

In [101]:
from sklearn.metrics import classification_report

probs_val = model.predict(X_val)
y_pred = (probs_val >= 0.5).astype(int)

print(classification_report(y_val, y_pred, digits=4))


              precision    recall  f1-score   support

           0     0.6269    1.0000    0.7706      7698
           1     0.0000    0.0000    0.0000      4582

    accuracy                         0.6269     12280
   macro avg     0.3134    0.5000    0.3853     12280
weighted avg     0.3930    0.6269    0.4831     12280



c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
